cosyvoice2

In [1]:
import sys
sys.path.append('/root/LLMs/CosyVoice')
sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')


from hyperpyyaml import load_hyperpyyaml
from cosyvoice.cli.frontend import CosyVoiceFrontEnd
import os
from cosyvoice.cli.model import CosyVoice2Model
from cosyvoice.utils.file_utils import load_wav
import torchaudio


model_dir = "/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})
frontend = CosyVoiceFrontEnd(configs['get_tokenizer'],
                                          configs['feat_extractor'],
                                          '{}/campplus.onnx'.format(model_dir),
                                          '{}/speech_tokenizer_v2.onnx'.format(model_dir),
                                          '{}/spk2info.pt'.format(model_dir),
                                          configs['allowed_special'])

model = CosyVoice2Model(configs['llm'], configs['flow'], configs['hift'], fp16=False)

model.load('{}/llm.pt'.format(model_dir),
           '{}/flow.pt'.format(model_dir),
           '{}/hift.pt'.format(model_dir))


failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-01-06 21:53:19,076 INFO input frame rate=25
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


提取 speaker emb linear layer 权重

In [3]:
model.flow.spk_embed_affine_layer

Linear(in_features=192, out_features=80, bias=True)

In [4]:
import torch

torch.save(model.flow.spk_embed_affine_layer, '/root/LLMs/models/cosyvoice/spk_embed_affine_layer.pt')

In [1]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

import torch
from flow_inference import AudioDecoder
import os.path


flow_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

sys.path.insert(0, "/root/LLMs/GLM-4-Voice/cosyvoice")
sys.path.insert(0, "/root/LLMs/CosyVoice/third_party/Matcha-TTS")

# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)

import uuid
# 生成唯一标识符
this_uuid = str(uuid.uuid4())

/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/_jit_internal.py:739: FutureWarning: ignore(True) has been deprecated. TorchScript will now drop the function call on compilation. Use torch.jit.unused now. {}
  warnings.warn(
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weigh

In [4]:
audio_decoder.flow.spk_embed_affine_layer = torch.load('/root/LLMs/models/cosyvoice/spk_embed_affine_layer.pt')

In [5]:
from speech_tokenizer.modeling_whisper import WhisperVQEncoder
from transformers import WhisperFeatureExtractor
from speech_tokenizer.utils import extract_speech_token

# 提取音频中的 token
input_speech_16k_path = ['/root/LLMs/CosyVoice/data/ZH_B00000_S00000_W000000.mp3']

# Speech tokenizer
whisper_model = (WhisperVQEncoder.from_pretrained(tokenizer_path).eval().to(device))

feature_extractor = WhisperFeatureExtractor.from_pretrained(tokenizer_path)

# 提取音频中的 token
audio_tokens = extract_speech_token(whisper_model, feature_extractor, input_speech_16k_path)[0]
audio_tokens = torch.tensor(audio_tokens).unsqueeze(0).to('cuda')

In [6]:
import torch
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
from matcha.utils.audio import mel_spectrogram

# sampling_rate = 22050

class Feat_extractor:

    def __init__(self, n_fft: int = 1920, num_mels: int = 80, sampling_rate: int = 24000, 
                 hop_size: int = 480, win_size: int = 1920, fmin: int = 0, fmax: int = 8000, 
                 center: bool = False, device: str = 'cuda'):
        self.device = torch.device(device)
        self.feat_extractor = lambda x: mel_spectrogram(
            x, n_fft=n_fft, num_mels=num_mels, sampling_rate=sampling_rate, 
            hop_size=hop_size, win_size=win_size, fmin=fmin, fmax=fmax, center=center
        )

    def __call__(self, speech):
        speech_feat = self.feat_extractor(speech).squeeze(dim=0).transpose(0, 1).to(self.device)
        speech_feat = speech_feat.unsqueeze(dim=0)
        speech_feat_len = torch.tensor([speech_feat.shape[1]], dtype=torch.int32).to(self.device)
        return speech_feat, speech_feat_len

import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import onnxruntime
import numpy as np

def extract_embedding(audio_path, onnx_model_path):
    """
    提取单个音频文件的 embedding
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: embedding (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 计算音频的FBank特征
    feat = kaldi.fbank(
        audio,
        num_mel_bins=80,  # 梅尔频带数
        dither=0,         # 禁用抖动
        sample_frequency=16000
    )

    # 特征归一化
    feat = feat - feat.mean(dim=0, keepdim=True)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 使用ONNX模型进行推理
    input_name = ort_session.get_inputs()[0].name
    embedding = ort_session.run(None, {input_name: feat.unsqueeze(dim=0).cpu().numpy()})[0]

    # 返回 embedding
    return embedding.flatten()

In [7]:
import sys
sys.path.append('/root/LLMs/CosyVoice')

# import uuid
from cosyvoice.utils.file_utils import load_wav
import torchaudio

voice_tokenizer_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/speech_tokenizer_v2.onnx'
embedding_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/campplus.onnx'
# this_uuid = str(uuid.uuid1())
# model.hift_cache_dict[this_uuid] = None

resample_rate = 22050

input_speech_16k_path = '/root/LLMs/CosyVoice/data/ZH_B00000_S00000_W000000.mp3'
input_speech_16k = load_wav(input_speech_16k_path, 16000)
# input_text = '希望你以后能够做的比我还好呦。'

# 提取 mel 特征
front_end = Feat_extractor()

input_speech_resample = torchaudio.transforms.Resample(orig_freq=32000, new_freq=resample_rate)(input_speech_16k)
input_feat, input_feat_len = front_end(input_speech_resample)

# 提取 speaker embedding
embedding = extract_embedding(input_speech_16k_path, embedding_path)
embedding = torch.tensor(embedding).unsqueeze(0).to('cuda')

In [11]:
# 生成语音和梅尔特征
with torch.no_grad():
    # 转换为张量
    tts_speech, tts_mel = audio_decoder.token2wav(
        token=audio_tokens,
        prompt_token = audio_tokens,
        uuid=this_uuid,
        prompt_feat=input_feat,
        embedding=embedding,
        finalize=True
    )

import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050
tts_speech = tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)